# <center>**I. DATA PRE-PROCESSING**

In [1]:
#Importing all the Necessary Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.simplefilter("ignore", UserWarning)


## <center>**survey_1 Data Preprocessing**

**Importing the raw dataset from a local directory into the survey_1 DataFrame.**

In [2]:
#Reading the file 
survey_1=pd.read_csv(r"C:\Users\shmes\OneDrive\schema_1_ontario_final.csv")

**1.To ensure each respondent is represented only once, we filtered the dataset to include only records where is_most_recent equals 1, effectively removing all duplicate historical submissions from the same individual.**

In [3]:
# --- STEP1: RETAIN ONLY MOST RECENT RECORDS ---

# 1. Filter the dataframe
# We keep rows where is_most_recent is 'y' (String match)
survey_1 = survey_1[survey_1['is_most_recent'] == 'y'].copy()

# 2. Drop the column
# We drop it here so it doesn't cause issues during the final merge/datatype conversion
survey_1 = survey_1.drop(columns=['is_most_recent'])

# 3. Verification
print(f"Remaining records in survey_1: {len(survey_1)}")

Remaining records in survey_1: 238945


**2.We addressed these names to move away from the raw, abbreviated column headers and toward a professional dataset structure. By renaming variables to specific terms like international_travel_history and known_covid_exposure, we ensure that every column has a precise meaning. This reduces the chance of error during complex analysis and ensures that all generated insights and charts are clear, specific, and ready for further reporting.**

In [4]:
# --- STEP 2:Renaming columns for better readability
# Survey_1 column renaming
rename_map = {
    'week': 'survey_week',
    'fsa': 'postal_fsa',
    'probable': 'is_probable_case',
    'vulnerable': 'is_vulnerable',
    'fever_chills_shakes': 'has_fever_chills_shakes',
    'cough': 'has_cough',
    'shortness_of_breath': 'has_shortness_of_breath',
    'over_60': 'is_age_over_60',
    'any_medical_conditions': 'has_premedical_conditions',
    'travel_outside_canada': 'international_travel',
    'contact_with_illness': 'known_covid_exposure'
}

survey_1.rename(columns=rename_map, inplace=True)

# Print the summary of the dataframe to check new column names and data types
print(survey_1.info())

<class 'pandas.core.frame.DataFrame'>
Index: 238945 entries, 0 to 263638
Data columns (total 11 columns):
 #   Column                     Non-Null Count   Dtype 
---  ------                     --------------   ----- 
 0   survey_week                238945 non-null  int64 
 1   postal_fsa                 238945 non-null  object
 2   is_probable_case           238945 non-null  object
 3   is_vulnerable              238945 non-null  object
 4   has_fever_chills_shakes    238945 non-null  object
 5   has_cough                  238945 non-null  object
 6   has_shortness_of_breath    238945 non-null  object
 7   is_age_over_60             238945 non-null  object
 8   has_premedical_conditions  238945 non-null  object
 9   international_travel       238945 non-null  object
 10  known_covid_exposure       238945 non-null  object
dtypes: int64(1), object(10)
memory usage: 21.9+ MB
None


**3.We cleaned the postal_fsa column by converting all entries to strings, capitalizing them, and stripping whitespace. This uniformity is critical for accurately grouping data by region and linking it to census datasets.**

In [5]:
# --- STEP 3: GEOGRAPHIC STANDARDIZATION ---
# Reasoning: Ensuring Canadian FSA codes are uniform (uppercase and no spaces) 
# is essential for reliable regional grouping and census data linkage.
survey_1['postal_fsa'] = survey_1['postal_fsa'].astype(str).str.upper().str.strip()

**4.We transformed qualitative 'y/n' variables (such as symptoms and exposure history) into quantitative binary data (1 and 0) to facilitate mathematical analysis.**

In [6]:
# --- STEP 4: BINARY MAPPING ---
# Reasoning: Converting text ('y/n') to numbers (1/0) enables mathematical operations 
# like calculating percentages. Any typos or empty cells automatically turn into NaNs.
binary_map = {'y': 1, 'n': 0}
binary_cols = [
    'is_probable_case', 'is_vulnerable', 'has_fever_chills_shakes', 
    'has_cough', 'has_shortness_of_breath', 'is_age_over_60', 
    'has_premedical_conditions', 'international_travel', 'known_covid_exposure'
]

for col in binary_cols:
    survey_1[col] = survey_1[col].map(binary_map)

**5.Missing values (`NaN`) were replaced with `-1`, assuming that a non-response indicates the absence of a symptom or condition. This conservative approach also allows conversion of columns to integer types, making the dataset ready for analysis while preserving information about non-responses.**

In [7]:
# --- STEP 5: HANDLING MISSING VALUES ---
# Reasoning: We replace NaNs with -1, assuming a non-response indicates the absence 
# of a symptom. This is a conservative approach that also allows for integer conversion.
survey_1[binary_cols] = survey_1[binary_cols].fillna(-1)

**6.Datatypes were explicitly assigned to transform raw survey entries into computable formats. This step ensures that numerical variables are available for statistical analysis and categorical variables are optimized for logical operations, preventing processing errors and ensuring the precision of all downstream calculations.**

In [8]:
# --- STEP 6: DATATYPE OPTIMIZATION ---
# Reasoning: We address these datatypes to improve performance: 
# int8 reduces memory use, bool speeds up filtering, and string ensures reliable text matching.
survey_1 = survey_1.astype({
    'survey_week': 'int8',
    'postal_fsa': 'string',
})
# Finalize binary columns to int8 (including is_most_recent)
for col in binary_cols:
    survey_1[col] = survey_1[col].astype('int8')

# <center>**survey_2 Data Preprocessing**

**Importing the raw dataset from a local directory into the survey_2 DataFrame.**

In [9]:
#Reading the file 
survey_2=pd.read_csv(r"C:\Users\shmes\OneDrive\schema_2_ontario_final.csv")

**1.To ensure each respondent is represented only once, we filtered the dataset to include only records where is_most_recent equals 1, effectively removing all duplicate historical submissions from the same individual.**

In [10]:
# --- STEP1: RETAIN ONLY MOST RECENT RECORDS ---

# 1. Filter the dataframe
# We keep rows where is_most_recent is 'y' (String match)
survey_2 = survey_2[survey_2['is_most_recent'] == 'y'].copy()

# 2. Drop the column
# We drop it here so it doesn't cause issues during the final merge/datatype conversion
survey_2 = survey_2.drop(columns=['is_most_recent'])

# 3. Verification
print(f"Remaining records in survey_2: {len(survey_2)}")

Remaining records in survey_2: 11723


**2.Column renaming to improve clarity, consistency, and readability of variable names for easier data interpretation and analysis.**

In [11]:
# --- STEP 2:Renaming columns for better readability
# Survey_2 column renaming.
rename_map = {
    'week': 'survey_week',
    'fsa': 'postal_fsa',
    'probable': 'is_probable_case',
    'vulnerable': 'is_vulnerable',
    'fever_chills_shakes': 'has_fever_chills_shakes',
    'cough': 'has_cough',
    'shortness_of_breath': 'has_shortness_of_breath',
    'any_medical_conditions': 'has_premedical_conditions',
    'travel_outside_canada': 'international_travel',
    'contact_with_illness': 'known_covid_exposure',
    'needs':'care_needs',
    'age_1' : 'age_group',
    'sex': 'gender'
}

survey_2.rename(columns=rename_map, inplace=True)

# Print the summary of the dataframe to check new column names and data types
print(survey_2.info())

<class 'pandas.core.frame.DataFrame'>
Index: 11723 entries, 0 to 14931
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   survey_week                11723 non-null  int64 
 1   postal_fsa                 11723 non-null  object
 2   is_probable_case           11723 non-null  object
 3   is_vulnerable              11723 non-null  object
 4   has_fever_chills_shakes    11723 non-null  object
 5   has_cough                  11723 non-null  object
 6   has_shortness_of_breath    11723 non-null  object
 7   has_premedical_conditions  11723 non-null  object
 8   international_travel       11723 non-null  object
 9   known_covid_exposure       11723 non-null  object
 10  symptoms                   1383 non-null   object
 11  conditions                 3577 non-null   object
 12  ethnicity                  5732 non-null   object
 13  gender                     11723 non-null  object
 14  care_needs 

**3.We cleaned the postal_fsa column by converting all entries to strings, capitalizing them, and stripping whitespace. This uniformity is critical for accurately grouping data by region and linking it to census datasets.**

In [12]:
# --- STEP 3: GEOGRAPHIC STANDARDIZATION ---
# Reasoning: Ensuring Canadian FSA codes are uniform (uppercase and no spaces) 
# is essential for reliable regional grouping and census data linkage.
survey_2['postal_fsa'] = survey_2['postal_fsa'].astype(str).str.upper().str.strip()

**4.We transformed qualitative 'y/n' variables (such as symptoms and exposure history) into quantitative binary data (1 and 0) to facilitate mathematical analysis.**

In [13]:
# --- STEP 4: BINARY MAPPING ---
# Reasoning: Converting text ('y/n') to numbers (1/0) enables mathematical operations 
binary_cols = ['is_probable_case', 'is_vulnerable', 'has_fever_chills_shakes', 
    'has_cough', 'has_shortness_of_breath','has_premedical_conditions', 'international_travel', 'known_covid_exposure']

**5.Missing values (`NaN`) were replaced with `-1`, assuming that a non-response indicates the absence of a symptom or condition. This conservative approach also allows conversion of columns to integer types, making the dataset ready for analysis while preserving information about non-responses.**

In [14]:
# --- STEP 5: HANDLING MISSING VALUES ---
# Reasoning: We replace NaNs with -1, assuming a non-response indicates the absence 
# of a symptom. This is a conservative approach that also allows for integer conversion.
for col in binary_cols:
    survey_2[col] = survey_2[col].map({'y': 1, 'n': 0}).fillna(-1)

**6.We applied a cleaning function, clean_to_snake_array, to standardize values in the Survey 2 dataset. It specifically targets the symptoms and conditions columns, converting text from various formats (like camelCase or spaced strings) into a uniform snake_case list format. This process ensures data consistency by handling missing values as none_reported and removing duplicate delimiters, which is essential for accurate merging and analysis across different surveys.**

In [15]:
# --- STEP 6: APPLY SNAKE_CASE ARRAY CLEANING TO SURVEY 2 ---
# Reasoning: Standardizing the internal list values to snake_case ensures 
# that Survey 2 and Survey 3 can be merged and "exploded" for analysis without 
# naming conflicts (e.g., 'feverChills' vs 'fever_chills').

import re
# 1. Define the Cleaning Function
def clean_to_snake_array(val):
    # Handle missing or empty values based on your current strategy
    if pd.isna(val) or val == "" or val == "none":
        return ["none_reported"]
    
    # If already a list, convert back to string for standard re-processing
    if isinstance(val, list):
        val = ";".join(val)
        
    # Split by semicolon (Flatten dataset delimiter)
    items = [item.strip() for item in str(val).split(';') if item.strip()]
    
    snake_items = []
    for s in items:
        # A. Split camelCase (e.g., 'stoppedTravelling' -> 'stopped_Travelling')
        # This identifies transitions between lowercase and uppercase
        s = re.sub(r'([a-z])([A-Z])', r'\1_\2', s)
        
        # B. Replace spaces and hyphens with underscores
        s = s.replace(' ', '_').replace('-', '_')
        
        # C. Remove duplicate underscores and lowercase everything
        s = re.sub(r'_+', '_', s).lower().strip('_')
        
        if s:
            snake_items.append(s)
            
    return snake_items if snake_items else ["none_reported"]

# 2. Define target columns for Survey 2
s2_target_columns = ['symptoms', 'conditions']

# 3. Apply the transformation
for col in s2_target_columns:
    if col in survey_2.columns:
        survey_2[col] = survey_2[col].apply(clean_to_snake_array)


**7.We standardized the gender labels to ensure consistent grouping and accurate statistical counts. By filling missing values with 'Unknown,' we preserved the total sample size and prevented data loss during analysis.**

In [16]:
# --- STEP 7: Gender Standardization
# Define the standard mapping
sex_standardization = {
    'm': 'Male',
    'f': 'Female',
    'Unknown': 'Unknown'
}

# Apply the mapping
survey_2['gender'] = survey_2['gender'].map(sex_standardization).fillna('Unknown')


**8.We standardized ethnicity labels to align inconsistent naming conventions across the surveys 2 and 3, ensuring a sufficient sample size for each group. This allows for a statistically accurate analysis of health disparities and targeted risk assessment across the entire study population.**

In [17]:
# --- STEP 8: COMPREHENSIVE ETHNICITY STANDARDIZATION ---

# 1. Pre-cleaning: remove whitespace and handle case
survey_2["ethnicity"] = survey_2["ethnicity"].astype(str).str.strip().str.lower()

# 2. Comprehensive Mapping Dictionary
# This covers the standard census categories often used in these health datasets
mapping = {
    # Caucasians
    "caucasian": "White/Caucasian",
    "white": "White/Caucasian",
    
    # Asian groups
    "asian": "Asian",
    "south asian": "South Asian",
    "southeast asian": "Southeast Asian",
    "east asian": "East Asian",
    "chinese": "East Asian",
    
    # Other major groups
    "black": "Black",
    "african": "Black",
    "latin american": "Latin American",
    "hispanic": "Latin American",
    "indigenous": "Indigenous",
    "arab": "Arab/Middle Eastern",
    "middle eastern": "Arab/Middle Eastern",
    "mixed": "Other/Mixed",
    "other": "Other/Mixed",
    
    # Missing/Unknown indicators
    "na": "Unknown",
    "nan": "Unknown",
    "none": "Unknown",
    "prefer not to say": "Unknown",
    "unknown": "Unknown",
    "": "Unknown"
}

# 3. Apply the mapping
# Using .map() + .fillna("Other/Unknown") ensures that if a value 
# isn't in our dictionary, it still gets a clean label.
survey_2["ethnicity"] = survey_2["ethnicity"].map(mapping).fillna("Unknown")

**9.Care needs were standardized into uniform categories to ensure comparability across the survey waves. This process transforms inconsistent descriptive responses into a structured "Support Scale," enabling a precise analysis of resource distribution and allowing for the identification of demographic groups with unmet medical and social needs.**

In [18]:
# --- STEP 9: --- STANDARDIZING CARE NEEDS ---
# Reasoning: Replacing NaNs with 'not_reported' to ensure consistent 
# categorical labels for descriptive analysis and visualization.

# 1. Fill blanks with 'not_reported'
survey_2['care_needs'] = survey_2['care_needs'].fillna('not_reported')

# 2. Convert to string type (to match your optimized schema)
survey_2['care_needs'] = survey_2['care_needs'].astype('string')

import re

def to_snake_case(text):
    if pd.isna(text) or text == 'not_reported':
        return text
    # Insert underscore before capital letters and lowercase
    return re.sub(r'(?<!^)(?=[A-Z])', '_', str(text)).lower()

# Apply to the column
survey_2['care_needs'] = survey_2['care_needs'].apply(to_snake_case)

**10.Datatypes were explicitly assigned to transform raw survey entries into computable formats. This step ensures that numerical variables are available for statistical analysis and categorical variables are optimized for logical operations, preventing processing errors and ensuring the precision of all downstream calculations.**

In [19]:
# --- STEP 10:
# 1. Define the columns that contain lists
list_cols = ['symptoms', 'conditions']
# 2. Convert lists to strings
for col in list_cols:
    # This checks if the value is a list; if so, it joins them with a comma
    # e.g., ['Fever', 'Cough'] becomes "Fever, Cough"
    survey_2[col] = survey_2[col].apply(lambda x: ', '.join(x) if isinstance(x, list) else x)
# 3. NOW you can safely convert them to categories without error
categorical_cols = [
    'postal_fsa', 'symptoms', 'conditions',
    'ethnicity', 'gender', 'care_needs', 'age_group'
]
for col in categorical_cols:
    survey_2[col] = survey_2[col].astype('category')

**11.Standardizing Integer Columns and converts them to the nullable Int8 data type, ensuring consistent, memory-efficient storage while safely handling missing values across key clinical and exposure indicators.**

In [20]:
# --- STEP 11:
# List of all remaining integer columns 
int_cols = [
    'survey_week', 'is_probable_case', 'is_vulnerable', 
    'has_fever_chills_shakes', 'has_cough', 'has_shortness_of_breath', 
    'has_premedical_conditions', 'international_travel', 'known_covid_exposure'
]

# Convert all to int8
for col in int_cols:
    survey_2[col] = survey_2[col].astype('Int8')

# <center>**survey_3 Data Preprocessing**

**Importing the raw dataset from a local directory into the survey_3 DataFrame.**

In [21]:
#Reading the file
survey_3=pd.read_csv(r"C:\Users\shmes\OneDrive\schema_3_ontario_final.csv")

**1.Column renaming to improve clarity, consistency, and readability of variable names for easier data interpretation and analysis.**

In [22]:
# --- STEP 2: Renaming for better readability
# Survey_3 column renaming
survey_3=pd.read_csv(r"C:\Users\shmes\OneDrive\schema_3_ontario_final.csv")
rename_map = {
    'month': 'survey_month',
    'fsa': 'postal_fsa',
    'probable': 'is_probable_case',
    'vulnerable': 'is_vulnerable',
    'fever_chills_shakes': 'has_fever_chills_shakes',
    'cough': 'has_cough',
    'shortness_of_breath': 'has_shortness_of_breath',
    'any_medical_conditions': 'has_premedical_conditions',
    'travel_outside_canada': 'international_travel',
    'contact_with_illness': 'known_covid_exposure',
    'tested' : 'covid_tested',
    'covid_positive' : 'covid_test_result',
    'needs':'care_needs',
    'age_1':'age_group',
    'mental_health_impact':'mental_health_status',
    'travel_work_school':'work_travel_status',
    'self_isolating':'isolation_status',
    'financial_obligations_impact':'financial_hardships',
    'tobacco_usage':'tobacco_use',
    'sex':'gender'
    
}

survey_3.rename(columns=rename_map, inplace=True)

# Print the summary of the dataframe to check new column names and data types
print(survey_3.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15534 entries, 0 to 15533
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   survey_month               15534 non-null  object
 1   postal_fsa                 15534 non-null  object
 2   is_probable_case           15534 non-null  object
 3   is_vulnerable              15534 non-null  object
 4   has_fever_chills_shakes    15534 non-null  object
 5   has_cough                  15534 non-null  object
 6   has_shortness_of_breath    15534 non-null  object
 7   has_premedical_conditions  15534 non-null  object
 8   international_travel       15534 non-null  object
 9   known_covid_exposure       15534 non-null  object
 10  contact_in_household       159 non-null    object
 11  covid_tested               15534 non-null  object
 12  covid_results_date         421 non-null    object
 13  covid_test_result          490 non-null    object
 14  sympto

**3.We cleaned the postal_fsa column by converting all entries to strings, capitalizing them, and stripping whitespace. This uniformity is critical for accurately grouping data by region and linking it to census datasets.**

In [23]:
# --- STEP 2: GEOGRAPHIC STANDARDIZATION ---
# Reasoning: Ensuring Canadian FSA codes are uniform (uppercase and no spaces) 
# is essential for reliable regional grouping and census data linkage.
survey_3['postal_fsa'] = survey_3['postal_fsa'].astype(str).str.upper().str.strip()

**4.We transformed qualitative 'y/n' variables (such as symptoms and exposure history) into quantitative binary data (1 and 0) to facilitate mathematical analysis.Missing values (`NaN`) were replaced with `-1`, assuming that a non-response indicates the absence of a symptom or condition. This conservative approach also allows conversion of columns to integer types, making the dataset ready for analysis while preserving information about non-responses.**

In [24]:
# --- STEP 3: BINARY MAPPING ---
# Reasoning: Converting text ('y/n') to numbers (1/0) enables mathematical operations 
# like calculating percentages. Any typos or empty cells automatically turn into NaNs.
binary_map = {'y': 1, 'n': 0}
binary_cols = [
    'is_probable_case', 'is_vulnerable', 'has_fever_chills_shakes', 
    'has_cough', 'has_shortness_of_breath', 
    'has_premedical_conditions', 'international_travel', 'known_covid_exposure','covid_tested','isolation_status'
]

for col in binary_cols:
    survey_3[col] = survey_3[col].map(binary_map)

# --- STEP 4: HANDLING MISSING VALUES ---
# Reasoning: We replace NaNs with -1, assuming a non-response indicates the absence 
# of a symptom. This is a conservative approach that also allows for integer conversion.
survey_3[binary_cols] = survey_3[binary_cols].fillna(-1)

**5.We applied a cleaning function, clean_to_snake_array, to standardize values in the Survey 2 dataset. It specifically targets the symptoms and conditions columns, converting text from various formats (like camelCase or spaced strings) into a uniform snake_case list format. This process ensures data consistency by handling missing values as none_reported and removing duplicate delimiters, which is essential for accurate merging and analysis across different surveys.**

In [25]:

# 1. Define the Cleaning Function
def clean_to_snake_array(val):
    # Handle missing or empty values based on your current strategy
    if pd.isna(val) or val == "" or val == "none":
        return ["none_reported"]
    
    # If already a list, convert back to string for standard re-processing
    if isinstance(val, list):
        val = ";".join(val)
        
    # Split by semicolon (Flatten dataset delimiter)
    items = [item.strip() for item in str(val).split(';') if item.strip()]
    
    snake_items = []
    for s in items:
        # A. Split camelCase (e.g., 'stoppedTravelling' -> 'stopped_Travelling')
        # This identifies transitions between lowercase and uppercase
        s = re.sub(r'([a-z])([A-Z])', r'\1_\2', s)
        
        # B. Replace spaces and hyphens with underscores
        s = s.replace(' ', '_').replace('-', '_')
        
        # C. Remove duplicate underscores and lowercase everything
        s = re.sub(r'_+', '_', s).lower().strip('_')
        
        if s:
            snake_items.append(s)
            
    return snake_items if snake_items else ["none_reported"]

# 2. Rename columns in survey_3 to match your request
# Based on your notebook headers
rename_map = {
    'needs': 'care_needs',
    'financial_obligations_impact': 'financial_hardships',
    'travel_work_school': 'work_travel_status'#,
    #'mental_health_impact': 'mental_health_status'
}
survey_3.rename(columns=rename_map, inplace=True)

# 3. Apply the transformation to ALL requested columns
target_columns = [
    'symptoms', 'conditions', 'care_needs', 'financial_hardships', 
    'media_channels', 'work_travel_status'#, 'mental_health_status'
]

for col in target_columns:
    if col in survey_3.columns:
        survey_3[col] = survey_3[col].apply(clean_to_snake_array)


**6.We standardized the gender labels to ensure consistent grouping and accurate statistical counts. By filling missing values with 'Unknown,' we preserved the total sample size and prevented data loss during analysis.**

In [26]:
# Define the standard mapping
sex_standardization = {
    'm': 'Male',
    'f': 'Female',
    'Unknown': 'Unknown'
}

# Apply the mapping
survey_3['gender'] = survey_3['gender'].map(sex_standardization).fillna('Unknown')

**7.Tobacco use categories were standardized to align inconsistent behavioral labels across survey waves into a uniform scale. Null values were explicitly filled with 'Not Reported' to prevent analytical bias, ensuring that missing data is treated as a distinct category rather than being incorrectly absorbed into the 'Non-smoker' group.**

In [27]:
# --- STEP 6: STANDARDIZING CATEGORICAL COLUMNS (TOBACCO USE) ---
# Reasoning: Unlike binary symptoms, we treat tobacco use as a multi-label category.
# We replace NaNs with 'not_reported' to maintain a human-readable format 
# suitable for descriptive analysis and charts.

# 1. Define the mapping to make labels more descriptive
tobacco_labels = {
    'n': 'non_smoker',
    'quitSmoking': 'quit_smoking',
    'y': 'smoker'
}

# 2. Apply labels and fill missing values
survey_3['tobacco_use'] = survey_3['tobacco_use'].replace(tobacco_labels)
survey_3['tobacco_use'] = survey_3['tobacco_use'].fillna('not_reported')

# 3. Cast to 'string' for memory efficiency and reliable text matching
survey_3['tobacco_use'] = survey_3['tobacco_use'].astype('string')

**8.We standardized the covid_test_result field by mapping raw values to clear labels (e.g., 'negatively', 'Negative') into a single categorical scale and assigning “Not Tested” to missing entries. This ensures that infection rates are calculated accurately and prevents the exclusion of critical case data during statistical modeling.**

In [28]:
# 1. Define the mapping for standardization
result_mapping = {
    'negatively': 'Negative',
    'n': 'Negative',
    'positively': 'Positive'
}

# 2. Apply mapping and fill missing values with 'Not Tested'
survey_3['covid_test_result'] = survey_3['covid_test_result'].map(result_mapping).fillna('Not Tested')

**9.We standardized the covid_test_result field by dropping schema-specific columns that are not required for downstream analysis to keep the dataset clean and consistent.**

In [29]:
# --- STEP: DROP UNNECESSARY COLUMNS ---

# 1. Define the columns to drop from survey_3
# These columns are specific to Schema 3
cols_to_drop_s3 = ['covid_results_date', 'contact_in_household']

# Apply the drop to survey_3
# axis=1 specifies columns; errors='ignore' prevents crashing if run twice
survey_3 = survey_3.drop(columns=cols_to_drop_s3, errors='ignore')

**10.Standardizing Mental Health Status Labels by mapping raw categorical values to clear, human-readable labels (Positive, Negative, No Impact) and assigning “Unknown” to missing entries, ensuring consistent interpretation for analysis and reporting.**

In [30]:

# 2. Define the Mapping
# Use a dictionary to map old values to clean, readable ones
status_map = {
    'negatively': 'Negative',
    'noImpact': 'No Impact',
    'positively': 'Positive'
}

# 3. Apply the Fix
# Map the values and fill any blanks with 'Unknown'
survey_3['mental_health_status'] = survey_3['mental_health_status'].map(status_map).fillna('Unknown')

**11.We standardized ethnicity labels through systematic cleaning and harmonization by handling missing or invalid entries, resolving capitalization issues, typos, multilingual variants, and multi-select responses, and mapping all values to a consistent set of standardized categories for reliable analysis.**

In [31]:
def standardize_ethnicity(value):
    # 1. FIX: Return 'unknown' if the value is missing (NaN) or empty
    if pd.isna(value) or value == '' or str(value).lower() == 'nan':
        return "unknown"
    # Safety check: if it's somehow a list, join it first
    if isinstance(value, list):
        value = ';'.join(value)
    if not isinstance(value, str):
        return "unknown"
    # 2. Define standard mappings (Comprehensive List)
    # Maps specific variations/typos/capitalization to the standard format
    mapping = {
        # French & Typos
        'hispanique': 'hispanic/latino',
        'premieres nations': 'firstNations',
        # Capitalization & Formatting Fixes
        'First Nations': 'firstNations',
        'first nations': 'firstNations',
        'White': 'caucasian',
        'white': 'caucasian',
        'Caucasian': 'caucasian',
        'East Asian': 'east asian',
        'South Asian': 'south asian',
        'Black/African': 'black/african',
        'Black': 'black/african',
        'Metis': 'metis',
        'Middle Eastern': 'middle eastern',
        'Indigenous': 'indigenous',
        'Latin American': 'hispanic/latino',
        'Hispanic': 'hispanic/latino',
        'Prefer not to answer': 'preferNotToAnswer',
        'prefer not to answer': 'preferNotToAnswer'
    }
    # 3. Split, Clean, Map
    parts = value.split(';')
    cleaned_parts = []
    for part in parts:
        part = part.strip()
        if not part: continue # Skip empty parts
        # Apply mapping: if the part is in the dict, replace it; otherwise keep original
        part = mapping.get(part, part)
        # Optional: Force lower case if not mapped, to ensure consistency?
        # part = part.lower()
        # (Only uncomment above line if you want EVERYTHING strictly lowercase)
        cleaned_parts.append(part)
    # 4. Handle case where all parts were empty
    if not cleaned_parts:
        return "unknown"
    # 5. Deduplicate and Sort
    unique_sorted_parts = sorted(list(set(cleaned_parts)))
    return ';'.join(unique_sorted_parts)
# --- Apply the function ---
survey_3['ethnicity'] = survey_3['ethnicity'].apply(standardize_ethnicity)

**12.We standardized data types across Survey 3 by converting cleaned binary and numeric indicator columns to efficient Int8 format, safely normalizing list-based fields into strings, and casting all categorical variables to the category data type to ensure consistency, memory efficiency, and reliable downstream analysis.**

In [32]:
# 1. Convert Integer/Float columns to int8 (Cleaned data only)
int_cols = [
    'is_probable_case', 'is_vulnerable', 'has_fever_chills_shakes',
    'has_cough', 'has_shortness_of_breath', 'has_premedical_conditions',
    'international_travel', 'known_covid_exposure', 'covid_tested', 'isolation_status'
]
for col in int_cols:
    survey_3[col] = survey_3[col].astype('int8')

# 2. Define ALL potential categorical columns
cat_cols = [
    'survey_month', 'postal_fsa', 'covid_test_result', 'symptoms', 
    'conditions', 'ethnicity', 'gender', 'care_needs', 'age_group', 
    'mental_health_status', 'work_travel_status', 'media_channels', 
    'financial_hardships', 'tobacco_use'
]

# 3. Clean LISTS out of ALL categorical columns first
for col in cat_cols:
    # This safer approach converts lists to strings and keeps existing strings as-is
    survey_3[col] = survey_3[col].apply(lambda x: ', '.join(map(str, x)) if isinstance(x, list) else x)

# 4. Now convert to category safely
for col in cat_cols:
    survey_3[col] = survey_3[col].astype('category')


**12.We created finalized, cleaned versions of all three survey datasets to clearly separate processed data from raw inputs and support reproducible downstream analysis.**

In [33]:
# Create the renamed versions of your cleaned datasets
survey_1_cleaned = survey_1
survey_2_cleaned = survey_2
survey_3_cleaned = survey_3

**13.We installed the required dependencies to enable reliable export of cleaned datasets into Excel and columnar storage formats.**

In [34]:
!pip install openpyxl

**14.We exported each cleaned survey dataset to separate Excel files with standardized sheet names, enabling easy sharing, validation, and review outside the Python environment.**

In [35]:
# Export Survey 1
survey_1_cleaned.to_excel("Covid_Survey_1_Python_2025.xlsx", sheet_name="Covid_Survey_1", index=False)

# Export Survey 2
survey_2_cleaned.to_excel("Covid_Survey_2_Python_2025.xlsx", sheet_name="Covid_Survey_2", index=False)

# Export Survey 3
survey_3_cleaned.to_excel("Covid_Survey_3_Python_2025.xlsx", sheet_name="Covid_Survey_3", index=False)

print("All three files have been successfully exported to your working directory.")

All three files have been successfully exported to your working directory.


**15.We created a dedicated directory to store all cleaned datasets, ensuring organized file management and a clear separation between raw and processed data.**

In [36]:
# Create a directory
import os
os.makedirs("cleaned_datasets", exist_ok=True)

**16.We saved all cleaned survey datasets in Parquet format using dtype-safe serialization, enabling efficient storage, faster loading, and seamless integration with analytics pipelines.**

In [37]:
#Save all 3 datasets (dtype-safe)
survey_1_cleaned.to_parquet(
    "cleaned_datasets/survey_1_cleaned.parquet",
    index=False
)
survey_2_cleaned.to_parquet(
    "cleaned_datasets/survey_2_cleaned.parquet",
    index=False
)
survey_3_cleaned.to_parquet(
    "cleaned_datasets/survey_3_cleaned.parquet",
    index=False
)

**17.We installed optimized Parquet engines to support high-performance columnar data storage and scalable data processing workflows.**

In [38]:
#Install dependency (only once)
!pip install pyarrow fastparquet

**18.We verified the final structure and data types of all cleaned datasets to confirm successful standardization, type safety, and readiness for statistical analysis and modeling.**

In [39]:
survey_1.info()
survey_2.info()
survey_3.info()

<class 'pandas.core.frame.DataFrame'>
Index: 238945 entries, 0 to 263638
Data columns (total 11 columns):
 #   Column                     Non-Null Count   Dtype 
---  ------                     --------------   ----- 
 0   survey_week                238945 non-null  int8  
 1   postal_fsa                 238945 non-null  string
 2   is_probable_case           238945 non-null  int8  
 3   is_vulnerable              238945 non-null  int8  
 4   has_fever_chills_shakes    238945 non-null  int8  
 5   has_cough                  238945 non-null  int8  
 6   has_shortness_of_breath    238945 non-null  int8  
 7   is_age_over_60             238945 non-null  int8  
 8   has_premedical_conditions  238945 non-null  int8  
 9   international_travel       238945 non-null  int8  
 10  known_covid_exposure       238945 non-null  int8  
dtypes: int8(10), string(1)
memory usage: 5.9 MB
<class 'pandas.core.frame.DataFrame'>
Index: 11723 entries, 0 to 14931
Data columns (total 16 columns):
 #   Column  